# Notebook 5 — Dashboard & Summary

**Objetivo:** visualizar los resultados clave del análisis mediante gráficos interactivos de Databricks.

| Visualización | Descripción |
|---------------|-------------|
| Top 10 ciudades | Volumen de transacciones por ciudad |
| Evolución temporal | Transacciones mensuales (top 10 ciudades) |
| Londres — tipos de propiedad | Volumen y precio por tipología |
| Londres — categorías de precio | Cuota de mercado por segmento |
| Londres — distritos (volumen) | Evolución mensual de los 10 distritos más activos |
| Londres — distritos (precio) | Evolución mensual de los 10 distritos más caros |

## Sección 1 — Ciudades: volumen y evolución temporal
Las dos visualizaciones siguientes muestran qué ciudades concentran más actividad y cómo ha evolucionado mes a mes.

In [0]:
WITH ranked_cities AS (
  SELECT
    town_city,
    SUM(total_transactions) AS total_transactions
  FROM
    workspace.uk_housing.gold_temporal_trends
  GROUP BY
    town_city
  ORDER BY
    total_transactions DESC
  LIMIT 10
)
SELECT
  CAST(CONCAT(year, '-', LPAD(month, 2, '0'), '-01') AS DATE) AS date_of_transfer,
  gtt.town_city,
  gtt.total_transactions,
  ROUND(gtt.avg_price, 0) AS avg_price,
  ROUND(gtt.std_price, 0) AS price_volatility,
  ROUND(gtt.total_value, 0) AS total_market_value
FROM
  workspace.uk_housing.gold_temporal_trends gtt
    INNER JOIN ranked_cities rc
      ON gtt.town_city = rc.town_city
ORDER BY
  date_of_transfer;

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

## Sección 2 — Londres: tipos de propiedad
Londres lidera el volumen nacional. Se analiza la composición de su mercado por tipo de vivienda.

In [0]:
SELECT 
    property_type_desc,
    total_transactions,
    ROUND(total_price, 0) AS total_market_value,
    ROUND(avg_price, 0) AS avg_price
FROM workspace.uk_housing.gold_property_analysis
WHERE town_city = 'LONDON'
ORDER BY total_transactions DESC;

## Sección 3 — Londres: segmentos de precio
Distribución del mercado londinense en los 4 segmentos. La window function calcula la cuota de cada segmento.

In [0]:
SELECT 
    price_category,
    total_transactions,
    ROUND(avg_price, 0) AS avg_price,
    ROUND(min_price, 0) AS min_price,
    ROUND(max_price, 0) AS max_price,
    ROUND((total_transactions * 100.0 / SUM(total_transactions) OVER()), 2) AS percentage_market
FROM workspace.uk_housing.gold_price_category_analysis
WHERE town_city = 'LONDON'
ORDER BY avg_price DESC;

## Sección 4 — Londres: distritos por volumen
Top 10 distritos con más transacciones. Permite ver qué zonas concentran la mayor actividad compradora.

In [0]:
WITH ranked_districts AS (
  SELECT
    district,
    SUM(total_transactions) AS total_transactions,
    AVG(avg_price) AS avg_price
  FROM
    workspace.uk_housing.gold_district_analysis
  WHERE town_city = 'LONDON'
  GROUP BY
    DISTRICT
  ORDER BY
    total_transactions DESC
  LIMIT 10
)
SELECT
  CAST(CONCAT(year, '-', LPAD(month, 2, '0'), '-01') AS DATE) AS date_of_transfer,
  gda.district,
  gda.total_transactions,
  ROUND(gda.avg_price, 0) AS avg_price,
  ROUND(gda.min_price, 0) AS min_price,
  ROUND(gda.max_price, 0) AS max_price
FROM
  workspace.uk_housing.gold_district_analysis gda
    INNER JOIN ranked_districts rd
      ON gda.district = rd.district
WHERE gda.town_city = 'LONDON'
ORDER BY
  date_of_transfer;

## Sección 5 — Londres: distritos por precio
Top 10 distritos más caros (filtrado >10 transacciones para evitar outliers estadísticos).

In [0]:
WITH ranked_districts AS (
  SELECT
    district,
    AVG(avg_price) AS avg_price,
    SUM(total_transactions) AS total_transactions
  FROM
    workspace.uk_housing.gold_district_analysis
  WHERE town_city = 'LONDON' AND total_transactions > 10
  GROUP BY
    DISTRICT
  ORDER BY
    avg_price DESC
  LIMIT 10
)
SELECT
  CAST(CONCAT(year, '-', LPAD(month, 2, '0'), '-01') AS DATE) AS date_of_transfer,
  gda.district,
  gda.total_transactions,
  ROUND(gda.avg_price, 0) AS avg_price,
  ROUND(gda.min_price, 0) AS min_price,
  ROUND(gda.max_price, 0) AS max_price
FROM
  workspace.uk_housing.gold_district_analysis gda
    INNER JOIN ranked_districts rd
      ON gda.district = rd.district
WHERE gda.town_city = 'LONDON'
ORDER BY
  date_of_transfer;